# 4. Özellik Çıkarımı (Feature Extraction)
Bu notebook, temizlenmiş veriden ML ve DL modelleri için özellik vektörleri (TF-IDF ve Sequence) oluşturur.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import gc
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import scipy.sparse

print('Veri yükleniyor...')
use_cols = ['label','cleaned_text','review_length','word_count',
            'exclamation_count','question_count','avg_word_length',
            'uppercase_ratio','sentiment_polarity','sentiment_subjectivity']
df = pd.read_csv('data/reviews_preprocessed.csv', usecols=use_cols)
df = df.dropna(subset=['cleaned_text'])
print(f'Veri yüklendi, Shape: {df.shape}')

Veri yükleniyor...


Veri yüklendi, Shape: (89267, 10)


## 4.1 TF-IDF (Klasik Modeller İçin)

In [2]:
print('TF-IDF dönüşümü uygulanıyor...')
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    dtype=np.float32
)
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])
print(f'TF-IDF Shape: {X_tfidf.shape}')

print('\nEn Sık 20 TF-IDF Özelliği:')
feat_names = np.array(tfidf_vectorizer.get_feature_names_out())
sums = X_tfidf.sum(axis=0).A1
top_idx = sums.argsort()[-20:][::-1]
for i, idx in enumerate(top_idx, 1):
    print(f'{i}. {feat_names[idx]} ({sums[idx]:.2f})')

TF-IDF dönüşümü uygulanıyor...


TF-IDF Shape: (89267, 50000)

En Sık 20 TF-IDF Özelliği:
1. not (2200.36)
2. like (1695.01)
3. taste (1661.81)
4. product (1513.02)
5. good (1461.24)
6. flavor (1364.48)
7. one (1296.38)
8. would (1173.80)
9. coffee (1162.81)
10. great (1123.54)
11. love (1065.31)
12. tea (997.52)
13. get (980.44)
14. no (926.01)
15. really (909.19)
16. much (899.12)
17. buy (881.47)
18. don (873.27)
19. time (858.92)
20. make (853.78)


In [3]:
print('Sayısal özellikler normalize ediliyor...')
numeric_features = [
    'review_length','word_count','exclamation_count','question_count',
    'avg_word_length','uppercase_ratio','sentiment_polarity','sentiment_subjectivity'
]
scaler = StandardScaler()
X_numeric = scaler.fit_transform(df[numeric_features])

print('TF-IDF ve sayısal özellikler birleştiriliyor...')
X_combined = scipy.sparse.hstack([X_tfidf, X_numeric]).tocsr()
print(f'Final Shape: {X_combined.shape}')

del X_tfidf, X_numeric, X_combined
gc.collect()

Sayısal özellikler normalize ediliyor...
TF-IDF ve sayısal özellikler birleştiriliyor...


Final Shape: (89267, 50008)


0

In [4]:
os.makedirs('models', exist_ok=True)
joblib.dump(tfidf_vectorizer, 'models/tfidf_vectorizer.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
print('TF-IDF ve Scaler kaydedildi.')

TF-IDF ve Scaler kaydedildi.


## 4.2 Sequence Encoding (Derin Öğrenme İçin)

In [5]:
print('Tokenizer eğitiliyor...')
NUM_WORDS = 50000
MAX_LEN = 200

word_counter = Counter()
for text in df['cleaned_text']:
    word_counter.update(str(text).split())

most_common = word_counter.most_common(NUM_WORDS - 1)
word_index = {word: i+1 for i, (word, _) in enumerate(most_common)}
vocab_size = len(word_index) + 1
print(f'Vocab Size: {vocab_size}')

tokenizer = {'word_index': word_index, 'num_words': NUM_WORDS}

Tokenizer eğitiliyor...


Vocab Size: 50000


In [6]:
print('Metinler sequence\'lara dönüştürülüyor...')
X_seq = np.zeros((len(df), MAX_LEN), dtype=np.int32)

for i, text in enumerate(df['cleaned_text']):
    words = str(text).split()
    seq = [word_index.get(w, 0) for w in words]
    length = min(len(seq), MAX_LEN)
    X_seq[i, :length] = seq[:length]

print(f'Sequence Shape: {X_seq.shape}')
for i in range(3):
    print(f'Örnek {i+1}: {X_seq[i][:15]}... (uzunluk: {len(X_seq[i])})')

Metinler sequence'lara dönüştürülüyor...


Sequence Shape: (89267, 200)
Örnek 1: [   3    2 1839  281  225   80  139 1839   80    3 1394   28 7905   60
  437]... (uzunluk: 200)
Örnek 2: [   3 3594 2269 2194    8  477  226   92   67  134 1699  409 1000 4754
 3626]... (uzunluk: 200)
Örnek 3: [ 29   1  10 844  15  10  14 613 280  98 483   0   0   0   0]... (uzunluk: 200)


In [7]:
os.makedirs('features', exist_ok=True)
joblib.dump(tokenizer, 'models/tokenizer.pkl')
np.save('features/sequences.npy', X_seq)
print('Sequence ve Tokenizer kaydedildi.')

Sequence ve Tokenizer kaydedildi.


## 4.3 Train / Validation / Test Split

In [8]:
print('Veri seti bölünüyor (%70 Train, %15 Val, %15 Test)...')
indices = np.arange(len(df))
y = df['label'].values

idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.30, random_state=42, stratify=y)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'\nEğitim (Train)     : {len(idx_train):,}')
print(f'Doğrulama (Val)    : {len(idx_val):,}')
print(f'Test               : {len(idx_test):,}')

def print_dist(y_sub, name):
    u, c = np.unique(y_sub, return_counts=True)
    d = {int(k): f'%{v/len(y_sub)*100:.1f}' for k, v in zip(u, c)}
    print(f'{name:10} Dağılım: {d}')

print_dist(y_train, 'Train')
print_dist(y_val, 'Val')
print_dist(y_test, 'Test')

Veri seti bölünüyor (%70 Train, %15 Val, %15 Test)...

Eğitim (Train)     : 62,486
Doğrulama (Val)    : 13,390
Test               : 13,391
Train      Dağılım: {0: '%33.3', 1: '%33.3', 2: '%33.3'}
Val        Dağılım: {0: '%33.3', 1: '%33.3', 2: '%33.3'}
Test       Dağılım: {0: '%33.3', 1: '%33.3', 2: '%33.3'}


In [9]:
np.save('features/train_idx.npy', idx_train)
np.save('features/val_idx.npy', idx_val)
np.save('features/test_idx.npy', idx_test)
np.save('features/labels.npy', y)
print('Split indeksleri kaydedildi.')

pd.DataFrame({
    'Set': ['Train','Validation','Test'],
    'Örnek Sayısı': [len(idx_train), len(idx_val), len(idx_test)],
    'Oran (%)': [len(idx_train)/len(y)*100, len(idx_val)/len(y)*100, len(idx_test)/len(y)*100]
})

Split indeksleri kaydedildi.


,Set,Örnek Sayısı,Oran (%)
0,Train,62486,69.998992
1,Validation,13390,14.999944
2,Test,13391,15.001064
